# SPINE Demo

This notebook demonstrates a compact same-slice workflow on SpatialGlue Dataset1 only:
1. Preprocess Dataset
2. Train and test on Dataset with a 9/1 split
3. Run standard training and display metrics

In [ ]:
from __future__ import annotations

import json
import shlex
import subprocess
import sys
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path

import torch

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'spine').exists() and (REPO_ROOT.parent / 'spine').exists():
    REPO_ROOT = REPO_ROOT.parent
RAW_DATA_ROOT = REPO_ROOT / 'Data_SpatialGlue'
PROJECT_ROOT = REPO_ROOT / 'spine'
TRAIN_SAVE_ROOT = PROJECT_ROOT / 'results_dir' / 'demo_same_slice'
DEVICE = 0
SEED = 42

DATASET_ID = 1
DATASET_SUFFIX = 'MINMAX_NOHVG_NOMAP'
DATASET_NAME = f'DATASET{DATASET_ID}_RNA_TO_PROTEIN_{DATASET_SUFFIX}'

TRAIN_EPOCHS = 500
BATCH_SIZE = 2
GRAD_ACC = 2
N_NEIGHBORS = 8
SAMPLE_TIMES = 10

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f'Using REPO_ROOT={REPO_ROOT}')
print(f'Using RAW_DATA_ROOT={RAW_DATA_ROOT}')
print(f'Using DATASET_NAME={DATASET_NAME}')
print(f'CUDA available={torch.cuda.is_available()}')

In [ ]:
preproc_path = REPO_ROOT / 'spine' / 'app' / 'preprocessing' / 'preprocessing.py'
spec = spec_from_file_location('spine_preprocessing', preproc_path)
preproc = module_from_spec(spec)
spec.loader.exec_module(preproc)

PYTHON = sys.executable

## 1. Preprocessing 

This cell preprocesses SpatialGlue Dataset1 only

In [ ]:
preproc.process_rna_to_protein_dataset(
    dataset_id=DATASET_ID,
    train_ratio=0.9,
    seed=SEED,
    dataset_suffix=DATASET_SUFFIX,
    save_pca_embeddings=True,
    pca_dim=256,
    project_root=str(PROJECT_ROOT),
    raw_data_root=str(RAW_DATA_ROOT),
)

## 2. Train/Test

This cell calls the existing CLI training entrypoint on Dataset1 using the standard training configuration.

In [ ]:
TRAIN_SAVE_ROOT.mkdir(parents=True, exist_ok=True)

cmd = [
    PYTHON,
    str(REPO_ROOT / 'spine' / 'app' / 'flow' / 'train_rna_to_protein.py'),
    '--dataset', DATASET_NAME,
    '--source_dataroot', str(PROJECT_ROOT / 'dataset'),
    '--embed_dataroot', str(PROJECT_ROOT / 'dataset' / 'embed_dataroot'),
    '--save_dir', str(TRAIN_SAVE_ROOT),
    '--exp_code', f'sameslice_d{DATASET_ID}_demo',
    '--seed', str(SEED),
    '--device', str(DEVICE),
    '--batch_size', str(BATCH_SIZE),
    '--gradient_accumulation_steps', str(GRAD_ACC),
    '--epochs', str(TRAIN_EPOCHS),
    '--sample_times', str(SAMPLE_TIMES),
    '--eval_step', '1',
    '--early_stop_metric', 'pcc',
    '--early_stop_patience', '50',
    '--n_neighbors', str(N_NEIGHBORS),
    '--gene_recon_weight', '0.2',
]

print(' '.join(shlex.quote(x) for x in cmd))
subprocess.run(cmd, check=True, cwd=str(REPO_ROOT))

## 3. Locate Latest Run Artifacts

In [ ]:
def find_latest_experiment(save_root: Path, prefix: str) -> Path:
    matches = sorted(p for p in save_root.glob(f'{prefix}_*') if p.is_dir())
    if not matches:
        raise FileNotFoundError(f'No experiment dirs found under {save_root} with prefix {prefix}')
    return matches[-1]

exp_prefix = f'sameslice_d{DATASET_ID}_demo'
latest_exp = find_latest_experiment(TRAIN_SAVE_ROOT, exp_prefix)
run_dir = latest_exp / DATASET_NAME
config_path = run_dir / 'config.json'
results_path = run_dir / 'split0' / 'results.json'
ckpt_path = run_dir / 'split0' / 'checkpoints' / 'best.pth'

print('latest_exp =', latest_exp)
print('config_path =', config_path)
print('results_path =', results_path)
print('ckpt_path =', ckpt_path)
assert config_path.exists(), config_path
assert results_path.exists(), results_path
assert ckpt_path.exists(), ckpt_path

## 4. Show Same-Slice Test Metrics

This cell reads `split0/results.json` and prints the final metrics from the training run.

In [ ]:
metrics = json.loads(results_path.read_text())

print('Same-slice Dataset1 test results')
print('PCC =', round(metrics['pearson_mean'], 4), '±', round(metrics.get('pearson_std', 0.0), 4))
print('MSE =', round(metrics['mse_mean'], 4))
print('MAE =', round(metrics['mae_mean'], 4))
print('saved at', results_path)